In [1]:
import numpy as np
import pandas as pd
import sqlite3
import statsmodels.formula.api as smf
from scipy.stats import chi2
from statsmodels.stats.multitest import multipletests

conn = sqlite3.connect('../data/hillstrom.db')
df = pd.read_sql("SELECT * FROM experiment", conn)

df['treatment_flag'] = (df['treatment'] == 'Womens E-Mail').astype(int)
df['recency_tier'] = pd.qcut(df['recency'], 3, labels=['Low', 'Mid', 'High'])

print(df['treatment_flag'].value_counts())
print()
print(df.groupby('recency_tier', observed=True)['recency'].agg(['min', 'max', 'count']))

treatment_flag
1    21387
0    21306
Name: count, dtype: int64

              min  max  count
recency_tier                 
Low             1    3  14924
Mid             4    8  14608
High            9   12  13161


In [2]:
def subgroup_lift(data, col):
    g = (data.groupby([col, 'treatment_flag'], observed=True)['visit']
             .agg(['mean', 'count']).unstack('treatment_flag'))
    out = pd.DataFrame({
        'n_control': g[('count', 0)],
        'n_treated': g[('count', 1)],
        'rate_control': g[('mean', 0)],
        'rate_treated': g[('mean', 1)],
    })
    out['lift_pp'] = (out['rate_treated'] - out['rate_control']) * 100
    return out.round(4)

for col in ['newbie', 'channel', 'recency_tier']:
    print(f"\n=== {col} ===")
    print(subgroup_lift(df, col))


=== newbie ===
        n_control  n_treated  rate_control  rate_treated  lift_pp
newbie                                                           
0           10611      10624        0.1337        0.1739   4.0123
1           10695      10763        0.0788        0.1292   5.0417

=== channel ===
              n_control  n_treated  rate_control  rate_treated  lift_pp
channel                                                                
Multichannel       2606       2579        0.1285        0.1756   4.7100
Phone              9327       9454        0.0872        0.1318   4.4630
Web                9373       9354        0.1189        0.1645   4.5677

=== recency_tier ===
              n_control  n_treated  rate_control  rate_treated  lift_pp
recency_tier                                                           
Low                7438       7486        0.1378        0.1818   4.4000
Mid                7339       7269        0.1022        0.1451   4.2943
High               6529       663

**Descriptive lifts.** New customers `newbie` show a larger absolute lift than existing ones (5.04 pp vs. 4.01 pp). Across `channel`, the three lifts sit within half a point of each other (4.46–4.71 pp), no visible difference. Across `recency_tier` (Low = purchased 1–3 months ago, High = 9–12 months ago), the High tier shows the largest lift (4.94 pp), while Low and Mid are close (4.40 pp and 4.29 pp). These are raw group means, before any significance testing.

In [3]:
def interaction_lr_test(data, moderator, covariates):
    """Likelihood-ratio test, does adding treatment x moderator improve the logit model?"""
    extra = (" + " + " + ".join(covariates)) if covariates else ""
    m_base = smf.logit(f"visit ~ treatment_flag + {moderator}{extra}", data=data).fit(disp=0)
    m_full = smf.logit(f"visit ~ treatment_flag * {moderator}{extra}", data=data).fit(disp=0)
    lr_stat = 2 * (m_full.llf - m_base.llf)
    dof = m_full.df_model - m_base.df_model
    p_value = chi2.sf(lr_stat, dof)
    return m_full, lr_stat, dof, p_value

print('='*50)
m_newbie, lr_newbie, dof_newbie, p_newbie = interaction_lr_test(df, 'newbie', ['recency', 'history'])
print(f"newbie: LR={lr_newbie:.3f}, df={int(dof_newbie)}, p={p_newbie:.4f}")
print(m_newbie.params.filter(like='treatment_flag').round(4))
print('='*50)
m_channel, lr_channel, dof_channel, p_channel = interaction_lr_test(df, 'C(channel)', ['recency', 'history'])
print(f"channel: LR={lr_channel:.3f}, df={int(dof_channel)}, p={p_channel:.4f}")
print(m_channel.params.filter(like='treatment_flag').round(4))
print('='*50)
m_tier, lr_tier, dof_tier, p_tier = interaction_lr_test(df, 'C(recency_tier)', ['history'])
print(f"recency_tier: LR={lr_tier:.3f}, df={int(dof_tier)}, p={p_tier:.4f}")
print(m_tier.params.filter(like='treatment_flag').round(4))
print('='*50)

newbie: LR=17.015, df=1, p=0.0000
treatment_flag           0.3106
treatment_flag:newbie    0.2479
dtype: float64
channel: LR=2.190, df=2, p=0.3345
treatment_flag                        0.3709
treatment_flag:C(channel)[T.Phone]    0.0959
treatment_flag:C(channel)[T.Web]      0.0081
dtype: float64
recency_tier: LR=10.096, df=2, p=0.0064
treatment_flag                            0.3267
treatment_flag:C(recency_tier)[T.Mid]     0.0741
treatment_flag:C(recency_tier)[T.High]    0.2377
dtype: float64


In [4]:
tests = ['newbie', 'channel', 'recency_tier']
p_raw = [p_newbie, p_channel, p_tier]
lr_stats = [lr_newbie, lr_channel, lr_tier]
dofs = [int(dof_newbie), int(dof_channel), int(dof_tier)]

reject, p_adj, _, _ = multipletests(p_raw, alpha=0.05, method='fdr_bh')

heterogeneity_table = pd.DataFrame({
    'Moderator': tests,
    'LR stat': np.round(lr_stats, 3),
    'df': dofs,
    'Raw p-value': p_raw,
    'FDR-adjusted p': p_adj,
    'Effect differs?': reject,
})
heterogeneity_table

,Moderator,LR stat,df,Raw p-value,FDR-adjusted p,Effect differs?
0,newbie,17.015,1,0.000037,0.000111,True
1,channel,2.190,2,0.334475,0.334475,False
2,recency_tier,10.096,2,0.006424,0.009636,True


**Heterogeneity verdict.** After FDR correction, the treatment effect differs significantly by `newbie` status (adj. p = 0.0001) and by `recency_tier` (adj. p = 0.0096), but not by `channel` (adj. p = 0.33). New customers respond more to the campaign than existing ones, and customers who haven't purchased recently (9–12 months) respond more than more recent purchasers, plausibly because these groups have more room to be moved by a nudge, whereas frequent buyers may visit regardless.

One caveat on `recency_tier`, the regression's interaction coefficients and the raw percentage-point lifts don't agree on ordering for the Mid tier (the interaction says Mid ≈ Low, but Mid's absolute lift is actually the lowest of the three). This happens because baseline visit rates fall steadily across tiers (13.8% → 10.2% → 7.5%), and the same log-odds shift produces a smaller absolute change at a lower baseline. Both readings are valid, although they answer slightly different questions.

Two of three tested dimensions already show real heterogeneity in coarse group averages. That's the empirical case for building an individual-level uplift model next, if the effect varies this much across a handful of buckets, it likely varies further, once recency, channel, and customer history are considered jointly rather than one at a time.